In [4]:
import numpy as np
from math import cos, acos, radians, degrees
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import patheffects as path_effects
import pickle

def create_vector_animation(all_results, params, nu_value, output_file, trail_length=5, fps=5, dpi=100, norm="inf", epsilon=1, numsteps=100, alpha2=2):
    # Extract steps and data
    steps = sorted([int(k.split('_')[1]) for k in all_results.keys()])
    grad_data = {step: all_results[f'step_{step}']['grad_coordinates'] for step in steps}
    delta_data = {step: all_results[f'step_{step}']['delta_coordinates'] for step in steps}
    
    # Extract loss data
    loss_data = {
        'with_solver': [all_results[f'step_{step}']['loss_metrics']['with_solver'] for step in steps],
        'detached': [all_results[f'step_{step}']['loss_metrics']['without_solver'] for step in steps],
        'approximated': [all_results[f'step_{step}']['loss_metrics']['approximate'] for step in steps]
    }

    angle_data = {
        'with_solver': [all_results[f'step_{step}']['grad_delta_withsolver'] for step in steps],
        'detached': [all_results[f'step_{step}']['grad_delta_detached'] for step in steps],
        'approximated': [all_results[f'step_{step}']['grad_delta_approximated'] for step in steps]
    }

    reach_boundary_data = {
        'with_solver': [all_results[f'step_{step}']['reaches_boundary_withsolver'] for step in steps],
        'detached': [all_results[f'step_{step}']['reaches_boundary_detached'] for step in steps],
        'approximated': [all_results[f'step_{step}']['reaches_boundary_approximated'] for step in steps]
    }

    # Create figure with 4 subplots
    fig = plt.figure(figsize=(24, 12))
    gs = fig.add_gridspec(2, 4)

    # Original plots (now in top row)
    ax1 = fig.add_subplot(gs[0, 0], projection='3d')  # Grad plot
    ax2 = fig.add_subplot(gs[0, 1], projection='3d')  # Delta plot
    ax3 = fig.add_subplot(gs[0, 2])  # Loss plot
    ax4 = fig.add_subplot(gs[0, 3])  # Angle plot

    # New comparison plots (bottom row)
    ax5 = fig.add_subplot(gs[1, 0])  # Input values
    ax6 = fig.add_subplot(gs[1, 1])  # FNO outputs
    ax7 = fig.add_subplot(gs[1, 2])  # PDE outputs

    # Set consistent limits for vector plots
    def get_max_range(data):
        max_val = 0
        for step in data.values():
            for vec in step.values():
                current_max = max(np.abs(vec))
                if current_max > max_val:
                    max_val = current_max
        return max_val * 0.9
    
    max_grad = get_max_range(grad_data)
    max_delta = get_max_range(delta_data)

    ax1.set_xlim([-max_grad, max_grad])
    ax1.set_ylim([-max_grad, max_grad])
    ax1.set_zlim([-max_grad, max_grad])
    ax2.set_xlim([-max_delta, max_delta])
    ax2.set_ylim([-max_delta, max_delta])
    ax2.set_zlim([-max_delta, max_delta])

    # Set up loss plot
    ax3.set_xlim([0, len(steps)])
    ax3.set_ylim([min(loss_data['with_solver']+loss_data['detached']+loss_data['approximated'])*0.95, 
                 max(loss_data['with_solver']+loss_data['detached']+loss_data['approximated'])*1.05])
    ax3.grid(True)
    ax3.set_xlabel('Step Number')
    ax3.set_ylabel('Loss')
    ax3.set_title('Loss Evolution')

    ax4.set_xlim([0, len(steps)])
    ax4.set_ylim([min(angle_data['with_solver']+angle_data['detached']+angle_data['approximated'])*0.95, 
                 max(angle_data['with_solver']+angle_data['detached']+angle_data['approximated'])*1.05])
    ax4.grid(True)
    ax4.set_xlabel('Step Number')
    ax4.set_ylabel('Angle')
    ax4.set_title('Angle Evolution between Delta and Gradient')

    # Color setup
    colors = {
        'with_solver': 'red',
        'without_solver': 'green',
        'detached': 'green', 
        'approximated': 'blue',
        'approximate': 'blue'
    }

    # Initialize loss lines
    loss_lines = {
        'with_solver': ax3.plot([], [], label='With Solver', color=colors['with_solver'])[0],
        'detached': ax3.plot([], [], label='Detached', color=colors['without_solver'])[0],
        'approximated': ax3.plot([], [], label='Approximated', color=colors['approximate'])[0]
    }
    ax3.legend()

    angle_lines = {
        'with_solver': ax4.plot([], [], label='With Solver', color=colors['with_solver'])[0],
        'detached': ax4.plot([], [], label='Detached', color=colors['without_solver'])[0],
        'approximated': ax4.plot([], [], label='Approximated', color=colors['approximate'])[0]
    }
    ax4.legend()

    def get_label_positions(max_val):
        return {
            'with_solver': (max_val*0.7, max_val*0.9, max_val*0.7),
            'detached': (max_val*0.7, max_val*0.6, max_val*0.7),
            'approximated': (max_val*0.7, max_val*0.3, max_val*0.7)
        }

    def get_label_positions_2(max_val):
        return {
            'with_solver': (-max_val*0.7, max_val*0.9, -max_val*0.7),
            'detached': (-max_val*0.7, max_val*0.6, -max_val*0.7),
            'approximated': (-max_val*0.7, max_val*0.3, -max_val*0.7)
        }

    grad_label_pos = get_label_positions(max_grad)
    delta_label_pos = get_label_positions_2(max_delta)
    
    """
    input_2 = data_dict['a_values']['a'].squeeze()
    input_with_solver = data_dict['a_values']['a_with'].squeeze()
    input_detached = data_dict['a_values']['a_detached'].squeeze()
    input_approximated = data_dict['a_values']['a_approximated'].squeeze()
    
    FNO = data_dict['G_values']['G(a)'].squeeze()
    FNO_with_solver = data_dict['G_values']['G(a_with)'].squeeze()
    FNO_detached = data_dict['G_values']['G(a_detached)'].squeeze()
    FNO_approximated = data_dict['G_values']['G(a_approximated)'].squeeze()
    
    PDE = data_dict['g_values']['g(a)']
    PDE_with_solver = data_dict['g_values']['g(a_with)']
    PDE_detached = data_dict['g_values']['g(a_detached)']
    PDE_approximated = data_dict['g_values']['g(a_approximated)']
    
    
    # Calculate y-axis limits for each category
    input_limits = (
        min(min(input_2), min(input_with_solver), min(input_detached), min(input_approximated)) * 0.95,
        max(max(input_2), max(input_with_solver), max(input_detached), max(input_approximated)) * 1.05
    )
    
    FNO_limits = (
        min(min(FNO), min(FNO_with_solver), min(FNO_detached), min(FNO_approximated)) * 0.95,
        max(max(FNO), max(FNO_with_solver), max(FNO_detached), max(FNO_approximated)) * 1.05
    )
    
    PDE_limits = (
        min(min(PDE), min(PDE_with_solver), min(PDE_detached), min(PDE_approximated)) * 0.95,
        max(max(PDE), max(PDE_with_solver), max(PDE_detached), max(PDE_approximated)) * 1.05
    )
    """

    def update(frame):
        step = steps[frame]
        
        for ax in [ax1, ax2, ax5, ax6, ax7]:
            for artist in ax.collections + ax.lines:
                artist.remove()
            for artist in ax.texts:
                artist.remove()
        
        # Add trail effect with transparency
        for i, hist_step in enumerate(steps[max(0, frame-trail_length):frame]):
            alpha = 0.2 * (i+1)/trail_length
            
            # Grad trails
            grad = grad_data[hist_step]
            for vec in grad:
                ax1.quiver(0, 0, 0, 
                          grad[vec][0], grad[vec][1], grad[vec][2], 
                          color=colors[vec], alpha=alpha, arrow_length_ratio=0.1)
            
            # Delta trails
            delta = delta_data[hist_step]
            for vec in delta:
                ax2.quiver(0, 0, 0, 
                          delta[vec][0], delta[vec][1], delta[vec][2], 
                          color=colors[vec], alpha=alpha, arrow_length_ratio=0.1)
        
        # Current frame vectors
        grad = grad_data[step]
        for vec in grad:
            ax1.quiver(0, 0, 0, 
                      grad[vec][0], grad[vec][1], grad[vec][2], 
                      color=colors[vec], arrow_length_ratio=0.1, linewidth=2)
            ax1.text(*grad_label_pos[vec], vec, color=colors[vec],
                    bbox=None, fontsize=15)
        
        delta = delta_data[step]
        for vec in delta:
            ax2.quiver(0, 0, 0, 
                      delta[vec][0], delta[vec][1], delta[vec][2], 
                      color=colors[vec], arrow_length_ratio=0.1, linewidth=2)

            if norm == 'inf':
                boundary_value = reach_boundary_data[vec][frame]  # a float in [0, 1]
                label_text = f"{vec} (reaches boundary={boundary_value*100:.1f}%)"
            elif norm == 2 or norm == "2":
                boundary_value = reach_boundary_data[vec][frame]  # a boolean
                label_text = f"{vec} (reaches boundary={'Yes' if boundary_value else 'No '})"
            else:
                label_text = vec  # fallback

            ax2.text(*delta_label_pos[vec], label_text, color=colors[vec],
                bbox=None, fontsize=15)
        
        # Update loss plot
        current_steps = steps[:frame+1]
        for key in loss_lines:
            loss_lines[key].set_data(current_steps, loss_data[key][:frame+1])
        for key in angle_lines:
            angle_lines[key].set_data(current_steps, angle_data[key][:frame+1])
        # Highlight current point in loss plot
        for key in ['with_solver', 'detached', 'approximated']:
            ax3.plot(step, loss_data[key][frame], '-', color=colors[key], markersize=8)
            ax4.plot(step, angle_data[key][frame], '-', color=colors[key], markersize=8)

        current_data = all_results[f'step_{step}']
        
        # Plot 1: Input values
        ax5.plot(current_data['a'], color='black', linestyle='-', label='Original', alpha=0.7)
        ax5.plot(current_data['a_with'], color='red', linestyle='-', label='With Solver', alpha=0.7)
        ax5.plot(current_data['a_detached'], color='green', linestyle='-', label='Detached', alpha=0.7)
        ax5.plot(current_data['a_approximated'], color='blue', linestyle='-', label='Approximated', alpha=0.7)
        ax5.set_title(f'Input Values (Step {step})')
        ax5.set_xlabel('Index')
        ax5.set_ylabel('Value')
        ax5.grid(True, alpha=0.3)
        ax5.legend(loc='upper right')
        
        # Plot 2: FNO outputs
        ax6.plot(current_data['G(a)'], color='black', linestyle='-', alpha=0.7, label='Original')
        ax6.plot(current_data['G(a_with)'], color='red', linestyle='-', alpha=0.7, label='With Solver')
        ax6.plot(current_data['G(a_detached)'], color='green', linestyle='-', alpha=0.7, label='Detached')
        ax6.plot(current_data['G(a_approximated)'], color='blue', linestyle='-', alpha=0.7, label='Approximated')
        ax6.set_title(f'FNO Outputs (Step {step})')
        ax6.set_xlabel('Index')
        ax6.set_ylabel('Value')
        ax6.grid(True, alpha=0.3)
        ax6.legend(loc='upper right')
        
        # Plot 3: PDE outputs
        ax7.plot(current_data['g(a)'], color='black', linestyle='-', alpha=0.7, label='Original')
        ax7.plot(current_data['g(a_with)'], color='red', linestyle='-', alpha=0.7, label='With Solver')
        ax7.plot(current_data['g(a_detached)'], color='green', linestyle='-', alpha=0.7, label='Detached')
        ax7.plot(current_data['g(a_approximated)'], color='blue', linestyle='-', alpha=0.7, label='Approximated')
        ax7.set_title(f'PDE Outputs (Step {step})')
        ax7.set_xlabel('Index')
        ax7.set_ylabel('Value')
        ax7.grid(True, alpha=0.3)
        ax7.legend(loc='upper right')
        
        param_title = f"ν={nu_value} | norm={norm} | steps={len(steps)} | ε={epsilon} | α={alpha2}"
        fig.suptitle(param_title, y=0.95)
        
        ax1.set_title(f'Gradient Vectors - Step {step}', pad=10)
        ax2.set_title(f'Delta Vectors - Step {step}', pad=10)
        ax3.set_title(f'Loss at Step {step}')
        ax4.set_title(f'Angle between Gradient and Delta at Step {step}')
        
        return []

    # Create animation
    ani = FuncAnimation(fig, update, frames=len(steps), interval=200, blit=False)

    # Save as GIF
    print(f"Creating animation and saving to {output_file}...")
    ani.save(output_file, writer='pillow', fps=fps, dpi=dpi)
    print("Animation saved successfully!")
    plt.close()
    
def compute_3d_coordinates(pairwise_data):
    """
    Compute 3D coordinates from pairwise comparisons.
    pairwise_data should contain:
    {
        'A_vs_B': {'A_norm': x, 'B_norm': y, 'cosine_similarity': z},
        'A_vs_C': {'A_norm': x, 'C_norm': y, 'cosine_similarity': z},
        'B_vs_C': {'B_norm': x, 'C_norm': y, 'cosine_similarity': z}
    }
    """
    try:
        # Extract all components from pairwise data
        A_norm = pairwise_data['A_vs_B']['A_norm']
        B_norm = pairwise_data['A_vs_B']['B_norm']
        C_norm = pairwise_data['A_vs_C']['C_norm']
        
        cos_AB = pairwise_data['A_vs_B']['cosine_similarity']
        cos_AC = pairwise_data['A_vs_C']['cosine_similarity']
        cos_BC = pairwise_data['B_vs_C']['cosine_similarity']
        
        # Build Gram matrix
        G = np.zeros((3,3))
        G[0,0] = A_norm**2
        G[1,1] = B_norm**2
        G[2,2] = C_norm**2
        G[0,1] = G[1,0] = A_norm * B_norm * cos_AB
        G[0,2] = G[2,0] = A_norm * C_norm * cos_AC
        G[1,2] = G[2,1] = B_norm * C_norm * cos_BC
        
        # Perform Cholesky decomposition
        try:
            L = np.linalg.cholesky(G)
            return {
                'with_solver': L[0].tolist(),    # A
                'detached': L[1].tolist(),       # B
                'approximated': L[2].tolist()    # C
            }
        except np.linalg.LinAlgError:
            # Fallback to eigenvalue decomposition
            w, v = np.linalg.eigh(G)
            L = v @ np.diag(np.sqrt(np.maximum(w, 0)))
            return {
                'with_solver': L[:,0].tolist(),
                'detached': L[:,1].tolist(),
                'approximated': L[:,2].tolist()
            }
            
    except KeyError as e:
        print(f"Missing required data: {e}")
        return None
        
def process_step(step_data):
    results = {}
    
    # Process grad_metrics if available
    if 'grad_metrics' in step_data:
        grad = step_data['grad_metrics']
        try:
            pairwise_data = {
                'A_vs_B': {
                    'A_norm': grad['withsolver_vs_detached']['gradient_with_solver_norm'],
                    'B_norm': grad['withsolver_vs_detached']['gradient_detached_norm'],
                    'cosine_similarity': grad['withsolver_vs_detached']['cosine_similarity']
                },
                'A_vs_C': {
                    'A_norm': grad['withsolver_vs_approximated']['gradient_with_solver_norm'],
                    'C_norm': grad['withsolver_vs_approximated']['gradient_approximated_norm'],
                    'cosine_similarity': grad['withsolver_vs_approximated']['cosine_similarity']
                },
                'B_vs_C': {
                    'B_norm': grad['detached_vs_approximated']['gradient_detached_norm'],
                    'C_norm': grad['detached_vs_approximated']['gradient_approximated_norm'],
                    'cosine_similarity': grad['detached_vs_approximated']['cosine_similarity']
                }
            }
            results['grad_coordinates'] = compute_3d_coordinates(pairwise_data)
        except KeyError as e:
            print(f"Missing grad_metrics data: {e}")
    
    # Process delta_metrics if available
    if 'delta_metrics' in step_data:
        delta = step_data['delta_metrics']
        try:
            pairwise_data = {
                'A_vs_B': {
                    'A_norm': delta['withsolver_vs_detached']['delta_with_solver_norm'],
                    'B_norm': delta['withsolver_vs_detached']['delta_detached_norm'],
                    'cosine_similarity': delta['withsolver_vs_detached']['cosine_similarity']
                },
                'A_vs_C': {
                    'A_norm': delta['withsolver_vs_approximated']['delta_with_solver_norm'],
                    'C_norm': delta['withsolver_vs_approximated']['delta_approximated_norm'],
                    'cosine_similarity': delta['withsolver_vs_approximated']['cosine_similarity']
                },
                'B_vs_C': {
                    'B_norm': delta['detached_vs_approximated']['delta_detached_norm'],
                    'C_norm': delta['detached_vs_approximated']['delta_approximated_norm'],
                    'cosine_similarity': delta['detached_vs_approximated']['cosine_similarity']
                }
            }
            results['delta_coordinates'] = compute_3d_coordinates(pairwise_data)
        except KeyError as e:
            print(f"Missing delta_metrics data: {e}")

    if 'loss_metrics' in step_data:
        results['loss_metrics'] = step_data['loss_metrics']

    results['grad_delta_withsolver'] = step_data['grad_delta_metrics']["withsolver"]['angle_degrees']
    results['grad_delta_detached'] = step_data['grad_delta_metrics']["detached"]['angle_degrees']
    results['grad_delta_approximated'] = step_data['grad_delta_metrics']["approximated"]['angle_degrees']

    results['reaches_boundary_withsolver'] = step_data['reaches_boundary']["with_solver"]
    results['reaches_boundary_detached'] = step_data['reaches_boundary']["detached"]
    results['reaches_boundary_approximated'] = step_data['reaches_boundary']["approximated"]

    results['a'] = step_data["values"]["a_values"]['a']
    results['a_with'] = step_data["values"]["a_values"]['a_with']
    results['a_detached'] = step_data["values"]["a_values"]['a_without']
    results['a_approximated'] = step_data["values"]["a_values"]['a_approximate']

    results['G(a)'] = step_data["values"]["G_values"]['G(a)']
    results['G(a_with)'] = step_data["values"]["G_values"]['G(a_with)']
    results['G(a_detached)'] = step_data["values"]["G_values"]['G(a_without)']
    results['G(a_approximated)'] = step_data["values"]["G_values"]['G(a_approximate)']

    results['g(a)'] = step_data["values"]["g_values"]['g(a)']
    results['g(a_with)'] = step_data["values"]["g_values"]['g(a_with)']
    results['g(a_detached)'] = step_data["values"]["g_values"]['g(a_without)']
    results['g(a_approximated)'] = step_data["values"]["g_values"]['g(a_approximate)']

    return results

def draw_gif(file_name,params):
    with open(file_name, 'rb') as file:
        data = pickle.load(file)
    nu_value = re.search(r'nu([\d.]+)', file_name).group(1)
    param_dict = {p.split('_')[0]: p.split('_')[1] for p in params}
    norm = param_dict.get('norm', 'inf')
    numsteps = param_dict.get('numsteps', '100')
    epsilon = param_dict.get('epsilon', '0.1')
    alpha2 = param_dict.get('alpha', '0.00500')

    output_file = f"grad_delta_nu{nu_value}_norm{norm}_steps{numsteps}_eps{epsilon}_alpha{alpha2}.gif"

    # Process all steps
    all_results = {}
    for step in range(1, int(numsteps)+1):
        step_key = f'step_{step}'
        if step_key in data[params]["step_update"]:  # Ensure 'data' is defined
            all_results[step_key] = process_step(data[params]["step_update"][step_key])

    create_vector_animation(all_results, params, nu_value, output_file,
                            trail_length=15, fps=10, 
                            norm=norm, numsteps=numsteps, epsilon=epsilon, alpha2=alpha2)

In [5]:
import re

file_name = 'gradient_comparison_exponax_nu0.0005_nsamples1.pkl'
params = ('norm_inf', 'index_0', 'numsteps_100', 'epsilon_0.1', 'alpha_0.00500')
draw_gif(file_name,params)

params = ('norm_2', 'index_0', 'numsteps_100', 'epsilon_10', 'alpha_0.50000')
draw_gif(file_name,params)

Creating animation and saving to grad_delta_nu0.0005_norminf_steps100_eps0.1_alpha0.00500.gif...
Animation saved successfully!
Creating animation and saving to grad_delta_nu0.0005_norm2_steps100_eps10_alpha0.50000.gif...
Animation saved successfully!


In [3]:
import re

file_name = 'gradient_comparison_exponax_nu0.0005_nsamples1.pkl'
params = ('norm_inf', 'index_0', 'numsteps_100', 'epsilon_0.1', 'alpha_0.00500')
# draw_gif(file_name,params)

params = ('norm_inf', 'index_0', 'numsteps_100', 'epsilon_0.5', 'alpha_0.02500')
draw_gif(file_name,params)

params = ('norm_inf', 'index_0', 'numsteps_100', 'epsilon_0.1', 'alpha_0.00200')
draw_gif(file_name,params)

params = ('norm_inf', 'index_0', 'numsteps_100', 'epsilon_0.5', 'alpha_0.01000')
draw_gif(file_name,params)

params = ('norm_2', 'index_0', 'numsteps_100', 'epsilon_10', 'alpha_0.50000')
# draw_gif(file_name,params)

params = ('norm_2', 'index_0', 'numsteps_100', 'epsilon_20', 'alpha_1.00000')
draw_gif(file_name,params)

params = ('norm_2', 'index_0', 'numsteps_100', 'epsilon_10', 'alpha_0.20000')
draw_gif(file_name,params)

params = ('norm_2', 'index_0', 'numsteps_100', 'epsilon_20', 'alpha_0.50000')
draw_gif(file_name,params)

Creating animation and saving to grad_delta_nu0.0005_norminf_steps100_eps0.5_alpha0.02500.gif...
Animation saved successfully!
Creating animation and saving to grad_delta_nu0.0005_norminf_steps100_eps0.1_alpha0.00200.gif...
Animation saved successfully!
Creating animation and saving to grad_delta_nu0.0005_norminf_steps100_eps0.5_alpha0.01000.gif...
Animation saved successfully!
Creating animation and saving to grad_delta_nu0.0005_norm2_steps100_eps20_alpha1.00000.gif...
Animation saved successfully!
Creating animation and saving to grad_delta_nu0.0005_norm2_steps100_eps10_alpha0.20000.gif...
Animation saved successfully!
Creating animation and saving to grad_delta_nu0.0005_norm2_steps100_eps20_alpha0.50000.gif...
Animation saved successfully!
